# 05 · Time-series foundation models (zero-shot)

Two pretrained forecasting models run **without any retraining** inside the same rolling-origin loop as the conventional models:

- **Chronos-Bolt Small** (Amazon) — a T5-style model trained on ~100B time-series observations; returns quantiles directly.
- **Moirai 2.0 R Small** (Salesforce) — a decoder-only quantile forecaster. Used in place of Google TimesFM: the TimesFM checkpoint (~800 MB) was too large to move into the project environment; Moirai 2.0 is a comparable zero-shot foundation model.

Each sees the last **512 weekly prices** at the origin (≈10 years), nothing else — no crude, no events, no dates. Median = point forecast; 10%/90% quantiles = 80% interval.

**Look-ahead check:** Chronos-Bolt was released in Nov 2024, before the test period. Moirai 2.0 was released in Aug 2025, about five weeks into the test period (repo last updated Jan 2026). Its pretraining corpus is built from public benchmark collections, so it is unlikely to contain this EIA series' test-period values, but its first few test weeks carry a small leakage risk, which we flag in the report. Weights were downloaded from Hugging Face into `models/` (files listed in the README).

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import *
import plots; plots.setup()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
df = load_weekly()
import time
from models_foundation import MODELS, MODEL_INFO
pd.DataFrame(MODEL_INFO).T

,name,params,weights,released,context
chronos_bolt,Chronos-Bolt Small (Amazon),48M,amazon/chronos-bolt-small,Nov 2024,512
moirai2,Moirai 2.0 R Small (Salesforce),11M,Salesforce/moirai-2.0-R-small,Aug 2025,512


In [2]:
ol = origins(df)
fcs = {}
for name, f in MODELS.items():
    t = time.time()
    fcs[name] = rolling_backtest(df, name, f, ol, verbose=False)
    print(f"{name:14s} done in {time.time() - t:6.1f}s")

Loading weights:   0%|          | 0/143 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 143/143 [00:00<00:00, 2848.62it/s]

chronos_bolt   done in    7.7s


Loading weights from local directory


moirai2        done in    2.5s


In [3]:
fc = attach_actuals(pd.concat(list(fcs.values()) + [pd.read_csv(FC_DIR / "naive.csv", parse_dates=["origin_date", "target_date"])]), df)
tab = fc[fc.horizon.isin(SCORED_H)].groupby(["horizon", "model"]).apply(metrics).round(3)
tab

n    MAE   RMSE    MAPE   bias  cover80
horizon model                                                   
4       chronos_bolt  63.0  0.277  0.384   5.308  0.174    0.683
        moirai2       63.0  0.279  0.389   5.296  0.172    0.730
        naive         63.0  0.270  0.384   5.233  0.111    0.794
12      chronos_bolt  55.0  0.579  0.829  10.592  0.453    0.691
        moirai2       55.0  0.639  0.877  11.717  0.533    0.709
        naive         55.0  0.600  0.812  11.109  0.294    0.782